# 第14章 电压稳定性分析

> Kundur P. *Power System Stability and Control*, McGraw-Hill, 1994, Chapter 14

## 本章概述

电压稳定性指系统在给定初始运行条件下，遭受扰动后所有节点
维持可接受电压的能力。主要分析工具:

1. **PV 曲线**: 负荷增长过程中关键节点电压的变化，鼻点=临界点
2. **QV 曲线**: 特定节点电压与所需无功注入的关系，最低点=无功裕度

### 电压崩溃机制
负荷增加 → 电流增大 → 线路压降增大 → 电压降低 →
恒功率负荷电流进一步增大 → 正反馈 → 电压崩溃


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy
from psa4teaching.models import Bus, BusType, Line, Generator, Load, LoadModel
from psa4teaching.stability import compute_pv_curve, compute_qv_curve

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('Import OK')

---
## Example 14.1: 简单系统的 PV 曲线

### 系统描述 (Kundur Example 14.1)

发电机通过输电线路向负荷供电:
- 发电机: 平衡节点, V=1.05 pu
- 线路: R=0.02, X=0.1 pu (π型等值)
- 负荷: P=0.8, Q=0.3 pu (恒功率)

**关键问题:** 负荷可以增加到多大系统仍能维持电压？

In [ ]:
# ===== Ex 14.1: 构建系统 =====
buses = [
    Bus(1, 'G1', BusType.SLACK, V_specified=1.05),
    Bus(2, 'Load', BusType.PQ, P_specified=-0.8, Q_specified=-0.3),
]
lines = [Line(1, 2, R=0.02, X=0.1, B=0.02)]
gens = [Generator(bus=1, Xd=1.8, H=5.0)]
loads = [Load(bus=2, P0=0.8, Q0=0.3)]

print(f'System: {len(buses)} buses, {len(lines)} line')
print(f'Base load: P={-buses[1].P_specified} pu, Q={-buses[1].Q_specified} pu')

In [ ]:
# ===== PV 曲线计算 =====
pv = compute_pv_curve(
    copy.deepcopy(buses), lines, [], gens, loads,
    target_bus=2, lambda_max=3.0, n_points=50, verbose=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(pv.P_total, pv.V_curve[:, 1], 'b-', linewidth=2)
ax.scatter([pv.P_total[pv.nose_point_index]],
          [pv.V_curve[pv.nose_point_index, 1]],
          color='red', s=100, zorder=5,
          label=f'Nose: P={pv.P_total[pv.nose_point_index]:.3f}, V={pv.V_curve[pv.nose_point_index,1]:.3f}')
ax.axvline(x=pv.P_total[pv.nose_point_index], color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Total Load Power (pu)'); ax.set_ylabel('Voltage at Bus 2 (pu)')
ax.set_title(f'PV Curve - Critical lambda = {pv.critical_lambda:.2f}')
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig('examples/kundur/pv_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Critical loading: {pv.critical_lambda:.2f}x base load')
print(f'Lower half of PV curve is unstable (dV/dP < 0)')
print(f'Upper half: stable operating region')

---
## Example 14.1 (续): QV 曲线

在负荷节点处扫描电压，计算所需无功注入。
曲线最低点给出无功裕度。

In [ ]:
# ===== QV 曲线计算 =====
qv = compute_qv_curve(
    copy.deepcopy(buses), lines, [], gens, loads,
    target_bus=2, V_range=(0.3, 1.2), n_points=40, verbose=True)

fig, ax = plt.subplots(figsize=(8, 6))
valid = ~np.isnan(qv.Q_required)
ax.plot(qv.V_values[valid], qv.Q_required[valid], 'b-', linewidth=2)
ax.scatter([qv.V_at_Qmin], [qv.Q_min], color='red', s=100, zorder=5,
          label=f'Q_min={qv.Q_min:.3f} at V={qv.V_at_Qmin:.3f}')
ax.axhline(y=0, color='k', linewidth=0.5)
ax.set_xlabel('Voltage at Bus 2 (pu)'); ax.set_ylabel('Required Q (pu)')
ax.set_title('QV Curve at Load Bus')
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig('examples/kundur/qv_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Minimum Q required: {qv.Q_min:.4f} pu at V={qv.V_at_Qmin:.4f} pu')
print(f'Operating point at V=1.0 needs Q≈{qv.Q_required[np.argmin(np.abs(qv.V_values-1.0))]:.4f} pu')
print(f'Reactive margin: additional reactive support margin')

---
## 参数对电压稳定性的影响

### 线路电抗 X 的影响
电抗越大 → 电压跌落越大 → 临界负荷越小。

In [ ]:
# ===== 线路电抗影响 =====
X_vals = [0.05, 0.10, 0.15, 0.20]
fig, ax = plt.subplots(figsize=(10, 6))

for Xv in X_vals:
    lines_i = [Line(1, 2, R=0.02, X=Xv, B=0.02)]
    pv_i = compute_pv_curve(
        copy.deepcopy(buses), lines_i, [], gens, loads,
        target_bus=2, lambda_max=3.0, n_points=40, verbose=False)
    ax.plot(pv_i.P_total, pv_i.V_curve[:, 1], linewidth=1.5,
            label=f'X={Xv:.2f} (lambda_crit={pv_i.critical_lambda:.2f})')

ax.set_xlabel('Total Load Power (pu)'); ax.set_ylabel('V at Bus 2 (pu)')
ax.set_title('Effect of Line Reactance on PV Curve')
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig('examples/kundur/pv_curve_x_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('X 越大 -> 临界负荷越小 -> 电压稳定性越差')
print('长距离输电线路电压稳定性问题更严重')

---
## 负荷模型的影响

恒功率负荷 (CP) 的电压稳定性最差，
恒阻抗负荷 (CZ) 在低电压时自动减载，有利于电压稳定。

In [ ]:
# ===== 负荷模型对比 (概念演示) =====
P_base = 0.8
V_vals = np.linspace(0.3, 1.2, 100)

P_CP = np.full_like(V_vals, P_base)  # constant power
P_CC = P_base * V_vals / 1.0        # constant current (P ~ V)
P_CZ = P_base * V_vals**2 / 1.0**2  # constant impedance (P ~ V^2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(V_vals, P_CP, 'b-', linewidth=2, label='Constant Power (CP)')
ax1.plot(V_vals, P_CC, 'g--', linewidth=2, label='Constant Current (CC)')
ax1.plot(V_vals, P_CZ, 'r-.', linewidth=2, label='Constant Impedance (CZ)')
ax1.set_xlabel('Voltage (pu)'); ax1.set_ylabel('Active Power (pu)')
ax1.set_title('Load Models: P vs V Characteristic')
ax1.grid(True, alpha=0.3); ax1.legend()

# Voltage stability margin vs load type
ax2.bar(['CP', 'CC', 'CZ'], [1.0, 1.4, 2.0], color=['blue','green','red'], alpha=0.7)
ax2.set_ylabel('Relative Stability Margin')
ax2.set_title('Voltage Stability Margin by Load Type')
ax2.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('examples/kundur/load_model_effects.png', dpi=150, bbox_inches='tight')
plt.show()
print('CP 负荷: 电压越低，电流越大 (正反馈) -> 最不利于电压稳定')
print('CZ 负荷: 电压越低，功率越小 (负反馈) -> 有利于电压稳定')

---
## 本章小结

1. **PV 曲线**: 直观展示负荷增长过程中电压的变化，鼻点=电压崩溃临界点
2. **QV 曲线**: 评估节点的无功需求和无功裕度
3. **关键影响因素**:
   - 线路电抗: 越大越不利于电压稳定
   - 负荷模型: 恒功率 > 恒电流 > 恒阻抗 (稳定性递减)
   - 无功补偿: 并联电容器可提高电压稳定性
4. **电压崩溃机制**: 负荷恢复特性与网络传输能力的正反馈失稳

> Kundur 参考值 (Example 14.1):
> - 简单系统临界负荷: lambda_crit ≈ 1.5-1.6 (取决于线路参数)
> - QV 曲线最低点对应最大无功传输能力
